### Reconstruct a streamed LLM response

An LLM sends its answer in fragments. Each fragment is a (sequence_number, text) pair. Fragments can arrive out of order, and a retry can deliver the same fragment twice.

Implement this Python function:
def assemble_response(fragments: list[tuple[int, str]]) -> str:
    ...

Requirements
- Sequence numbers are nonnegative and start at 0. Join the text in sequence order, preserving spaces exactly.
- If the same sequence number arrives twice with identical text, include it once.
- If the same sequence number arrives with different text, raise ValueError.
- If a sequence number is missing between 0 and the highest number received, raise ValueError. Empty input returns "".


fragments = [(2, "ready."), (0, "Your "), (2, "ready."), (1, "report is ")]

The expected result is "Your report is ready.". The second (2, "ready.") is a harmless retry. If it were (2, "pending."), the function would have to report a conflict.

First question: As you read the fragments one at a time, what would you store so that when sequence number 2 appears again, you can tell whether it’s a retry or a conflict?

In [ ]:
def assemble_response(fragments: list[tuple[int, str]]) -> str:
    if not fragments:
        return ""
    seen = {}
    
    for seq_no, text in fragments:
        if seq_no < 0:
            raise ValueError("Sequence numbers must be non-negative")
        if seq_no in seen:
            if seen[seq_no] != text:
                raise ValueError("Conflicting text for sequence {seq_no}")
            # Identical text already recorded; skip it
            continue
        seen[seq_no] = text

    # chck for missing sequence numbers between 0 and max(seen)
    max_seq = max(seen.keys())
    if len(seen) != max_seq + 1:
        raise ValueError("Missing sequence numbers")
    # join text in order from 0 to max_seq 
    return "".join(seen[i] for i in range(max_seq + 1))
      
# Usage 
fragments = [(2, "ready."), (0, "Your "), (2, "ready."), (1, "report is ")]
result = assemble_response(fragments)

print (result)


Your report is ready.


• "What is the time and space complexity of your solution?"
	• Time: O(N) to iterate through the N input fragments, O(U) to find max_seq across U unique keys, and O(U + L) to join the strings (where L is total string length). Overall: O(N + L) time.
	• Space: O(U + L) to store the seen dictionary and build the final joined string.

Quick Checklist
• Clarify constraints first: Ask about input size, negative numbers, thread safety, and whether inputs can be mutated.
• State complexity out loud: Mention time and space complexity before and after optimizing.
• Walk through an edge case: Manually trace an empty input, a duplicate, and an invalid input before saying "I'm done."